In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

In [ ]:
DATA_PATH = "./student_performance_dataset.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")

In [ ]:
print("Dataset Shape: ")

print(df.shape)

In [ ]:
# create column for binary divisor (in this analysis, did the student get a b or higher?
df['b_or_higher'] = np.where(df['final_grade'].isin(['A', 'B']), 1, 0)

In [ ]:
print("Dataset Info\n")
df.info()

In [ ]:
df.head()

In [ ]:
# shuffle dataframe and split into train and test sets

shuffled_df = df.sample(frac=1).reset_index(drop=True)

# define columns of interest and create df from it
xs, ys = "final_exam_score", "b_or_higher"
selected_columns = [xs, ys]
selected_df = shuffled_df[selected_columns]

# 80/20 split
split = int(0.8 * len(df)) 

# create training set
train = selected_df[:split]

# use z-scores normalization to make the slope and bias each move with similar significance with each step
# without it, 
train_mean = np.mean(train[xs])
train_std = np.std(train[xs])
train[xs] = ((train[xs] - train_mean) / train_std)

test = selected_df[split:]
test[xs] = ((test[xs] - train_mean) / train_std)

print(train.shape)
print(test.shape)

In [ ]:
#initialize random slope and bias to start

slope = random.uniform(-1, 1)
bias = random.uniform(-1, 1)

In [ ]:
# run gradient descent 1000 times

for i in range(1000):
    # create predictions array
    preds = train[xs] * slope + bias

    # squash preds into probabilities using sigmoid
    probs = 1 / (1 + np.exp(-preds)) 

    # ensure no probabilities round to 1 or 0:
    probs = np.clip(probs, 1e-7, 1 - 1e-7)

    # take log loss
    loss = -np.mean(train[ys] * np.log(probs) + (1 - train[ys]) * np.log(1 - probs))
    
    
    # take derivative of loss wrt. preds (referred to as z here)
    dldz = probs - train[ys]

    # derivative of preds wrt. slope
    dzds = train[xs]
    
    # derivative of loss wrt. slope
    dlds = dldz * dzds
    
    slope_grads = dlds # relabelled
    
    
    # derivative of preds wrt. bias
    dzdb = 1
    # derivative of loss wrt. bias
    dldb = dldz * dzdb
    bias_grads = dldb
    

    # take average of gradients
    
    slope_grad_avg = (sum(slope_grads)) / len(slope_grads)
    
    bias_grad_avg = (sum(bias_grads)) / len(bias_grads)
    
            
    # nudge slope and bias in direction opposite to gradient

    step_size = 0.01
    
    slope -= (slope_grad_avg * step_size)
    
    bias -= (bias_grad_avg * step_size)
    
    
print(loss)

In [ ]:
# training scoring

probs_binary = (probs > 0.5).astype(int)
probs_binary

count = 0

for i, j in zip(probs_binary, train[ys]):
    if i == j:
        count += 1

count

In [ ]:
np.sum(train[ys])

In [ ]:
plt.scatter(train[xs], train[ys])

sort_idx = np.argsort(train[xs])
plt.plot(train[xs].values[sort_idx], probs.values[sort_idx])

In [ ]:
test_preds = test[xs] * slope + bias

test_probs = 1 / (1 + np.exp(-test_preds)) 


plt.scatter(test[xs], test[ys])

sort_idx = np.argsort(test[xs])
plt.plot(test[xs].values[sort_idx], test_probs.values[sort_idx])

In [ ]:
# testing scoring 

test_probs_binary = (test_probs > 0.5).astype(int)
test_probs_binary

count = 0

for i, j in zip(test_probs_binary, test[ys]):
    if i == j:
        count += 1

count
